#Second Practical work

This is second practical work skeleton.

It is divided in 3 parts_

1.   Python module load section: load all the needed modules
2.   Google Drive access and mounting
3.   Host load data
4.   GPU load





##Module load section
We will install the pycuda module in the colaboratory environment, to combine python and CUDA in our code. Because pycuda is not available by default, this task can take a few minutes.

As well we load numpy and matplotlib modules.

In [ ]:
!pip install pycuda

We import the Python modules which we will use in the notebook (numpy, pycuda and time module) 

In [ ]:
import  numpy  as  np
import  pycuda.autoinit
from    pycuda.compiler import SourceModule
import  pycuda.driver as  drv
import  pycuda.gpuarray as  gpuarray
from sys import getsizeof
import time

In [ ]:
%pylab inline
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cProfile

## Mount Google Drive and access grant
We use Google API to allow access to our Google Drive.

For security and privacy reasons, this permission is temporally, and will be requested every time you restart the notebook. 

Your Google Drive will be mounted in the "/content/drive" folder. The first Folder will be "My Drive".

You have to copy your images to a folder in the "My Drive" folder in your Google Drive.

To check your available files, use the cell indicated as "**List Files in Folder**"

In [ ]:
# from os.path import join
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
# MYDRIVE="/content/drive/My Drive/"

**List Files in Folders**

In [ ]:
!ls /content/drive/My\ Drive/images

We define a variable IMAGE which is constructed joining the "MYDRIVE" varible plus the final path to your image.

In [ ]:
IMAGE=join(MYDRIVE,HERE YOU HAVE TO WRITE THE IMAGE TO LOAD)

In [ ]:
print("Image to load: {0}".format(IMAGE))

Loads an image in a numpy variable, and show.

In [ ]:
image=np.array(Image.open(IMAGE))
plt.figure()
plt.imshow(image)

#FILTER KERNEL

Here you have to define your `image_filter` GPU kernel.

* The kernel works on **one layer (channel) of the image**, a 2D matrix stored row by row: element (row, col) is at `row * width + col`.
* Block of 16 x 16 threads, one thread per output pixel. Grid: `ceil(width/16) x ceil(height/16)`.
* Use **tiled (shared) memory** with coalesced loads: a 2D shared array `tile[TILE][TILE]` (TILE = 16 + 5 - 1 = 20), indexed `tile[y][x]` as in numpy. Borders are replicated with `min`/`max`, without `if`.
* Copy the **filter mask to shared memory** too (`smask[5][5]`, indexed `smask[i][j]`), so the threads do not read it from global memory in every iteration.
* The kernel always computes the **correlation** and stores the raw sum: no clipping and no branches. Convolution, normalisation and clipping are done in Python.
* `filtered_img` is padded to `H16 x W16` (multiples of 16), so every thread writes without checking bounds. `W16 = gridDim.x * 16`.
* All integer parameters and indices are **signed** `int`.

In [ ]:
kernel = SourceModule("""
#define BLOCK    16                       // Block size: BLOCK x BLOCK threads, one thread per output pixel
#define MASK_MAX 5                        // Maximum mask size (5x5)
#define TILE     (BLOCK + MASK_MAX - 1)   // Tile side: 20 for BLOCK = 16

__global__ void image_filter(float *image,          // Source image LAYER (one channel), height x width, float32
                             float *filtered_img,   // Target layer, PADDED to H16 x W16 (multiples of BLOCK), float32
                             float *mask,           // Filter mask, mask_h x mask_w, float32 (row-major)
                             int width,             // Image number of columns
                             int height,            // Image number of rows
                             int mask_w,            // Mask number of columns (odd, <= 5)
                             int mask_h)            // Mask number of rows    (odd, <= 5)
{
    // YOU HAVE TO WRITE HERE YOUR FILTER KERNEL USING TILED (SHARED) MEMORY.
    //
    // Shared memory, 2D arrays indexed [row][column] as in numpy:
    //     __shared__ float tile[TILE][TILE];           // tile[y][x]: image block + halo
    //     __shared__ float smask[MASK_MAX][MASK_MAX];  // smask[i][j]: filter mask
    //   Used area: tile (BLOCK + mask_h - 1) rows x (BLOCK + mask_w - 1) columns, smask mask_h x mask_w.
    //
    //   1. Cooperative, coalesced load of the tile using the linear thread id
    //      t = threadIdx.x + BLOCK * threadIdx.y; replicate borders with min/max (no if).
    //   2. Copy the mask from global memory to smask with the same linear scheme.
    //   3. __syncthreads()  (one barrier for both loads)
    //   4. Correlation: sum of smask[i][j] * tile[threadIdx.y + i][threadIdx.x + j], in a register.
    //   5. Store the raw sum (no clipping) in filtered_img[row * W16 + col].
    // All the integer indices must be SIGNED (the tile origin can be negative).

    // THIS SKELETON CODE JUST COPIES THE SOURCE LAYER INTO THE PADDED TARGET LAYER.
    int col = blockIdx.x * BLOCK + threadIdx.x;     // Output column
    int row = blockIdx.y * BLOCK + threadIdx.y;     // Output row
    int W16 = gridDim.x * BLOCK;                    // Padded width of filtered_img

    int src_col = min(col, width  - 1);             // Replicate right border (no branch)
    int src_row = min(row, height - 1);             // Replicate bottom border (no branch)

    filtered_img[row * W16 + col] = image[src_row * width + src_col];
}
""")

And import the kernel definition to python. The Python handle is called `image_filter_gpu`, so it does not clash with the host function `image_filter`.

In [ ]:
image_filter_gpu = kernel.get_function('image_filter')

# HOST FUNCTION

`image_filter(image, filter, is_conv, norm)` prepares the data, decides correlation or convolution, invokes the GPU kernel once per layer and adjusts the output range.

* `image`: RGB image, integer values between 0 and 255.
* `filter`: filter mask, odd dimensions, at most 5x5.
* `is_conv`: 0 applies the formula as correlation, 1 as convolution.
* `norm`: 1 rescales the output linearly to 0..255, 0 saturates it.

It returns the filtered image as a NumPy array.

In [ ]:
BLOCK = 16   # Must match the BLOCK value defined in the kernel


def image_filter(image: np.ndarray,
                 filter: np.ndarray,
                 is_conv: int,
                 norm: int) -> np.ndarray:
    """
    Filters an RGB image on the GPU, one layer (channel) at a time.

    image   : RGB image, integer values between 0 and 255, shape (rows, cols, 3).
              An alpha channel, if present, is discarded.
    filter  : filter mask, odd dimensions, at most 5x5.
    is_conv : 0 -> correlation, 1 -> convolution (the mask is rotated 180 degrees here, on the host).
    norm    : 1 -> linear rescaling of the output to 0..255, 0 -> saturation (clipping) to 0..255.

    Returns the filtered image as a uint8 NumPy array with the same shape as the RGB input.
    """
    # 1. Input validation -------------------------------------------------------------
    if image.ndim != 3 or image.shape[2] < 3:
        raise ValueError("image must be an RGB array of shape (rows, cols, 3)")
    rgb = image[:, :, :3]                                   # Drop alpha channel if any
    mask = np.asarray(filter, dtype=np.float32)
    if mask.ndim != 2:
        raise ValueError("filter must be a 2D array")
    mask_h, mask_w = mask.shape
    if mask_h % 2 == 0 or mask_w % 2 == 0 or mask_h > 5 or mask_w > 5:
        raise ValueError("filter dimensions must be odd and at most 5x5")

    # 2. Correlation or convolution: decided HERE, so the kernel has no branch --------
    #    Convolution = correlation with the mask rotated 180 degrees
    #    (rows AND columns reversed; this is NOT the transpose).
    if is_conv:
        mask = mask[::-1, ::-1]
    mask = np.ascontiguousarray(mask, dtype=np.float32)    # The reversed view is not contiguous

    # 3. Mask normalisation: if the coefficients do not add up to 0 or 1 ---------------
    mask_sum = float(mask.sum())
    if not np.isclose(mask_sum, 0.0) and not np.isclose(mask_sum, 1.0):
        mask = (mask / mask_sum).astype(np.float32)

    # 4. Execution configuration and GPU buffers (reused for the 3 layers) -------------
    height, width = rgb.shape[:2]
    grid = (int(np.ceil(width / BLOCK)), int(np.ceil(height / BLOCK)), 1)
    block = (BLOCK, BLOCK, 1)
    W16, H16 = grid[0] * BLOCK, grid[1] * BLOCK             # Padded output size

    mask_gpu  = gpuarray.to_gpu(mask)
    layer_gpu = gpuarray.empty((height, width), np.float32)
    out_gpu   = gpuarray.empty((H16, W16), np.float32)

    result = np.empty((height, width, 3), dtype=np.float32)

    # 5. One kernel launch per layer ---------------------------------------------------
    for layer in range(3):
        image_layer = np.ascontiguousarray(rgb[:, :, layer], dtype=np.float32)  # rgb[:,:,c] is a strided view
        layer_gpu.set(image_layer)
        image_filter_gpu(layer_gpu, out_gpu, mask_gpu,
                         np.int32(width), np.int32(height),
                         np.int32(mask_w), np.int32(mask_h),
                         block=block, grid=grid)
        result[:, :, layer] = out_gpu.get()[:height, :width]  # Crop the padding

    # 6. Output range adjustment on the host (no conditionals in the kernel) -----------
    if norm:
        s_min, s_max = result.min(), result.max()             # Over the 3 layers together
        if s_max > s_min:
            result = 255.0 * (result - s_min) / (s_max - s_min)
        else:
            result = np.zeros_like(result)

    return np.clip(np.rint(result), 0, 255).astype(np.uint8)

Here we define our sample filter masks.

In [ ]:
filter1=np.array([
    [0,0,0,0,0],
    [0,0,0,0,0],
    [0,0,1,0,0],
    [0,0,0,0,0],
    [0,0,0,0,0]
])
filter2=np.array([[0.5, 0 , -0.5]])
filter3=np.array([[0.5],[0],[-0.5]])

filter4=np.array([
    [1,0,-1],
    [2,0,-2],
    [1,0,-1]
])
filter5=np.array([
    [0.00078633,0.00655965,0.01330373,0.00655965,0.00078633],
    [0.00655965,0.05472157,0.11098164,0.05472157,0.00655965],
    [0.01330373,0.11098164,0.22508352,0.11098164,0.01330373],
    [0.00655965,0.05472157,0.11098164,0.05472157,0.00655965],
    [0.00078633,0.00655965,0.01330373,0.00655965,0.00078633]
])

## Run the filter
Select the filter mask and the options, and call the host function.

In [ ]:
filter_mask = filter4     # Select the filter mask
IS_CONV = 0               # 0: correlation, 1: convolution
NORM = 1                  # 1: rescale output to 0..255, 0: saturate output

image_filter(image, filter1, 0, 0)   # Warm-up call (first launch includes CUDA context overhead)

start_t = time.time()
result_image = image_filter(image, filter_mask, IS_CONV, NORM)
end_t = time.time()

#Show results

In [ ]:
print("Execution time is: {0} seconds.".format(end_t - start_t))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(image[:, :, :3]); axes[0].set_title("Original")
axes[1].imshow(result_image);    axes[1].set_title("Filtered")
for ax in axes:
    ax.axis("off")
plt.show()